In [1]:
import duckdb

In [6]:
query = """
WITH CTE_jan_2023 AS (
    SELECT 
        tpep_pickup_datetime AS pick_up_time,
        tpep_dropoff_datetime AS drop_off_time,
        passenger_count,
        trip_distance,
        payment_type,
        total_amount,
        tip_amount
    FROM 'C:/Users/ekadw/Documents/DATA/NY_Taxi/2023/yellow_taxi/yellow_tripdata_2023-01.parquet'
), CTE_two AS (
SELECT
    *
FROM CTE_jan_2023
WHERE (passenger_count >= 0) AND (trip_distance >= 0) AND (total_amount >=0) AND (trip_distance <= 50) AND (tip_amount >0 )
), CTE_three AS (
SELECT
    *
FROM CTE_two
WHERE payment_type = 1
), CTE_four AS (
SELECT
    *,
    DATE_DIFF('day', pick_up_time, drop_off_time) AS duration_days 
FROM CTE_three
), CTE_five AS (
SELECT
    *
FROM CTE_four
WHERE duration_days = 0
), CTE_six AS (
SELECT
    *,
    DATE_DIFF('second', pick_up_time, drop_off_time) AS duration_seconds
FROM CTE_five
), CTE_seven AS (
SELECT
    *,
    CASE 
        WHEN tip_amount > 0 THEN 0
        ELSE 1
    END AS tip_category
FROM CTE_six
), CTE_eight AS (
SELECT
    passenger_count,
    trip_distance,
    total_amount,
    duration_seconds,
    tip_category
FROM CTE_seven
)

SELECT 
    *
FROM CTE_eight
LIMIT 100000
"""
#con = duckdb.connect()
#df = con.execute(query_yellow_jan_2023).fetchdf()
#df

In [9]:
import duckdb
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
import matplotlib.pyplot as plt
from sklearn.utils.class_weight import compute_class_weight
# ----------------------
# Config
# ----------------------
#query = """
#SELECT feature1, feature2, feature3, label
#FROM 'your_data/*.parquet'
#WHERE fare_amount > 0
#"""
# Compute weights from a sample of data
sample = con.execute(query).fetchdf()
weights = compute_class_weight('balanced', classes=classes, y=sample['tip_category'].to_numpy())
class_weights = {cls: w for cls, w in zip(classes, weights)}
print("Class Weights:", class_weights)

batch_size = 100000   # tune based on your memory
classes = np.array([0, 1])  # adjust if multi-class

# ----------------------
# Setup
# ----------------------
con = duckdb.connect()
scaler = StandardScaler()
clf = SGDClassifier(loss="log_loss", class_weight=class_weights, random_state=42)

# ----------------------
# Pass 1: fit scaler
# ----------------------
offset = 0
while True:
    batch = con.execute(f"{query} LIMIT {batch_size} OFFSET {offset}").fetchdf()
    if batch.empty:
        break
    X = batch.drop(columns=["tip_category"]).to_numpy()
    scaler.partial_fit(X)
    offset += batch_size

# ----------------------
# Pass 2: train model
# ----------------------
offset = 0
while True:
    batch = con.execute(f"{query} LIMIT {batch_size} OFFSET {offset}").fetchdf()
    if batch.empty:
        break
    X = batch.drop(columns=["tip_category"]).to_numpy()
    y = batch["tip_category"].to_numpy()
    X_scaled = scaler.transform(X)
    clf.partial_fit(X_scaled, y, classes=classes)
    offset += batch_size

# ----------------------
# Pass 3: evaluate streaming
# ----------------------
TP = FP = TN = FN = 0

# Histogram bins for ROC
n_bins = 1000
pos_hist = np.zeros(n_bins, dtype=np.int64)
neg_hist = np.zeros(n_bins, dtype=np.int64)

offset = 0
while True:
    batch = con.execute(f"{query} LIMIT {batch_size} OFFSET {offset}").fetchdf()
    if batch.empty:
        break
    X = batch.drop(columns=["tip_category"]).to_numpy()
    y = batch["tip_category"].to_numpy()
    X_scaled = scaler.transform(X)

    preds = clf.predict(X_scaled)
    probs = clf.predict_proba(X_scaled)[:, 1]

    # Update confusion matrix
    for yi, pi in zip(y, preds):
        if yi == 1 and pi == 1:
            TP += 1
        elif yi == 0 and pi == 1:
            FP += 1
        elif yi == 0 and pi == 0:
            TN += 1
        elif yi == 1 and pi == 0:
            FN += 1

    # Update histogram bins for ROC
    bins = np.floor(probs * (n_bins - 1)).astype(int)
    for yi, bi in zip(y, bins):
        if yi == 1:
            pos_hist[bi] += 1
        else:
            neg_hist[bi] += 1

    offset += batch_size

# ----------------------
# Final metrics
# ----------------------
accuracy  = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

# ----------------------
# Approximate ROC from histogram
# ----------------------
tpr, fpr = [], []
cum_pos, cum_neg = 0, 0
total_pos, total_neg = pos_hist.sum(), neg_hist.sum()

for i in reversed(range(n_bins)):  # thresholds from 1 → 0
    cum_pos += pos_hist[i]
    cum_neg += neg_hist[i]
    tpr.append(cum_pos / total_pos if total_pos > 0 else 0)
    fpr.append(cum_neg / total_neg if total_neg > 0 else 0)

roc_auc = np.trapz(tpr, fpr)

print(f"ROC AUC  : {roc_auc:.4f}")

# ----------------------
# Plot ROC
# ----------------------
plt.figure()
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (Approx. from histogram)")
plt.legend(loc="lower right")
plt.show()


ValueError: classes should have valid labels that are in y

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pillow-11.3.0-cp311-cp311-win_amd64.whl.metadata (9.2 kB)
  Using cached pyparsing-3.2.3-py3-none-any.whl.metadata (5.0 kB)
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.1 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.1 MB 2.2 MB/s eta 0:00:04
   ------ --------------------------------- 1.3/8.1 MB 2.3 MB/s eta 0:00:03
   --------- ------------------------------ 1.8/8.1 MB 2.4 MB/s eta 0:00:03
   ----------- ---------------------------- 2.4/8.1 MB 2.4 MB/s eta 0:00:03
   -------------- ------------------------- 2.9/8.1 MB 2.4 MB/s eta 0:00:03
   ---------------- ----------------------- 3.4/8.1 MB 2.5 MB/s eta 0:00:02
   ------------------- -------------------- 3.9/8.1 MB 2.5 MB/s eta 0:00:02
   --------------------- ------------------ 4.5/8.1 MB 2.5 MB/s eta 0:00:02
   ------------------------ --------------- 5.0/8.1

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.9 MB 1.2 MB/s eta 0:00:08
   --- ------------------------------------ 0.8/8.9 MB 1.2 MB/s eta 0:00:07
   ----- ---------------------------------- 1.3/8.9 MB 1.4 MB/s eta 0:00:06
   ------- -------------------------------- 1.6/8.9 MB 1.5 MB/s eta 0:00:05
   --------- ------------------------------ 2.1/8.9 MB 1.6 MB/s eta 0:00:05
   ----------- ---------------------------- 2.6/8.9 MB 1.8 MB/s eta 0:00:04
   --------------- ------------------------ 3.4/8.9 MB 1.9 MB/s eta 0:00:03
   ----------------- ---------------------- 3.9/8.9 MB 2.0 MB/s eta 0:00:03
   ------------------- -------------------- 4.5/8.9 MB 2.1 MB/s eta 0:00:03
   ---------------------- --------------

In [9]:
query_all = """
SELECT 
    *
FROM 'C:/Users/ekadw/Documents/DATA/NY_Taxi/2023/yellow_taxi/yellow_tripdata_2023-01.parquet'
"""
con = duckdb.connect()
df = con.execute(query_all).fetchdf()
df

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.30,1.00,0.5,0.00,0.0,1.0,14.30,2.5,0.00
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.90,1.00,0.5,4.00,0.0,1.0,16.90,2.5,0.00
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.90,1.00,0.5,15.00,0.0,1.0,34.90,2.5,0.00
3,1,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,1.0,N,138,7,1,12.10,7.25,0.5,0.00,0.0,1.0,20.85,0.0,1.25
4,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1.0,N,107,79,1,11.40,1.00,0.5,3.28,0.0,1.0,19.68,2.5,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3066761,2,2023-01-31 23:58:34,2023-02-01 00:12:33,NaN,3.05,NaN,None,107,48,0,15.80,0.00,0.5,3.96,0.0,1.0,23.76,NaN,NaN
3066762,2,2023-01-31 23:31:09,2023-01-31 23:50:36,NaN,5.80,NaN,None,112,75,0,22.43,0.00,0.5,2.64,0.0,1.0,29.07,NaN,NaN
3066763,2,2023-01-31 23:01:05,2023-01-31 23:25:36,NaN,4.67,NaN,None,114,239,0,17.61,0.00,0.5,5.32,0.0,1.0,26.93,NaN,NaN
3066764,2,2023-01-31 23:40:00,2023-01-31 23:53:00,NaN,3.15,NaN,None,230,79,0,18.15,0.00,0.5,4.43,0.0,1.0,26.58,NaN,NaN
